# Where does the shadow cross Earth?

An eclipse catalog gives us one row for an event that lasts several hours. For the classifier this is enough to label a day. For the trajectory we need to know where the shadow was at a particular instant and where the Sun and Moon were at that same instant. This distinction determines which rows we keep from each source.

Run the cells in a writable folder with internet access. The four CSV files produced here are the inputs to Data Cleaning.

In [ ]:
%pip install pandas requests beautifulsoup4

In [ ]:
import pandas as pd
import requests

## A · One row per eclipse

We start with the time of greatest eclipse. The catalog gives this time in terrestrial dynamical time and tabulates the difference from universal time in seconds. We keep both quantities because subtracting that difference can move the event across midnight. A partial eclipse also counts as an event. Its missing central duration is a consequence of the geometry.

The row count checks that the parser found all 11,898 entries. It cannot detect an incorrect longitude or time difference within an entry. NASA lists corrections to the printed catalog. The extraction below reads the ASCII values as supplied. [Catalog and errata](https://eclipse.gsfc.nasa.gov/5MCSE/5MKSEerrata.html)

In [ ]:
url_a = "https://eclipse.gsfc.nasa.gov/5MCSE/5MKSEcatalog.txt"

response = requests.get(url_a, timeout=60)
response.raise_for_status()

# 1. Define the columns
columns_a = [
    "cat_no", "canon_plate", "year", "month", "day",
    "td_ge", "dt", "luna_num", "saros", "eclipse_type",
    "qle", "gamma", "magnitude", "lat_ge", "lng_ge",
    "sun_alt", "sun_azm", "path_width", "central_duration"
]

# 2. Extract the eclipse records
rows = []

text = response.content.decode("latin-1")

for line in text.splitlines():
    values = line.split()

    if len(values) >= 17 and values[0].isdigit(): #There are some columns with missing values we do this procedure to make sure that all rows are included.
        if values[9][0] in ["P", "A", "T", "H"]:

            while len(values) < len(columns_a):
                values.append(None)

            rows.append(values[:len(columns_a)])

# 3. Create the DataFrame
df_a = pd.DataFrame(rows, columns=columns_a)

# 4. Convert numeric columns
numeric_a = [
    "cat_no", "canon_plate", "year", "day", "dt",
    "luna_num", "saros", "gamma", "magnitude",
    "sun_alt", "sun_azm", "path_width"
]

for col in numeric_a:
    df_a[col] = pd.to_numeric(df_a[col], errors="coerce") #We make sure that all the numerical columns correspond to a numeric type value.

# 5. Validate and save
print("Dataset shape:", df_a.shape)

if len(df_a) != 11898:
    raise ValueError("The catalog is incomplete.")

df_a.set_index("cat_no", inplace=True)
display(df_a.head())

df_a.to_csv("dataset_A.csv", index=False)

## B · A second description of the same events

The Besselian elements describe the position and evolution of the shadow relative to a fundamental plane. Here we preserve their coefficients in a separate table. Later we construct the axis from the Sun and Moon vectors, so these coefficients do not enter the fitted classifier or the physical trajectory.

In [ ]:
#We follow the same procedure to extract dataset B
# 1. Data source
url_b = "https://eclipse.gsfc.nasa.gov/eclipse_besselian_from_mysqldump2.csv"

# 2. Extract the data
df_b = pd.read_csv(url_b, low_memory=False)
# 3. Clean column names
df_b.columns = df_b.columns.str.strip()
# 4. Validate the dataset
print("Dataset shape:", df_b.shape)
if len(df_b) != 11898:
    raise ValueError("The Besselian catalog is incomplete.")
display(df_b.head())
# 5. Save the dataset
df_b.to_csv("dataset_B.csv", index=False)

## C · A path has a time coordinate

Pick a single row of a NASA path table. Its three pairs of geographic coordinates locate the northern limit, southern limit and center at one UT clock time. Each angle is stored in two pieces. For example, $12^\circ30'\,\mathrm{S}$ becomes $-(12+30/60)=-12.5^\circ$. A dash occupies a single field and means the position is unavailable. Treating it as zero would put a point on the equator or the Greenwich meridian.

In [ ]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# 1. Main NASA page
url_c = "https://eclipse.gsfc.nasa.gov/SEpath/SEpath.html"


# 2. Convert coordinates to decimal degrees
def read_coordinate(parts, i):

    if parts[i] == "-":
        return float("nan"), i + 1

    degrees = float(parts[i])
    minutes = float(parts[i + 1][:-1])
    direction = parts[i + 1][-1]

    if direction not in ["N", "S", "E", "W"]:
        raise ValueError("Invalid coordinate")

    coordinate = degrees + minutes / 60

    if direction in ["S", "W"]:
        coordinate = -coordinate

    return coordinate, i + 2


# 3. Extract the path of one eclipse
def extract_path(url):

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")

    event_id = url.split("/")[-1].replace("path.html", "")

    rows = []

    for pre in soup.find_all("pre"):

        for line in pre.get_text().splitlines():

            parts = line.split()
            if not parts:
                continue

            clock = parts[0]

            if len(clock) != 5 or clock[2] != ":":
                continue

            if not (clock[:2].isdigit() and clock[3:].isdigit()):
                continue

            try:
                coordinates = []
                i = 1

                for j in range(6):
                    value, i = read_coordinate(parts, i)
                    coordinates.append(value)

                extra = parts[i:]

                if len(extra) < 5:
                    continue

                row = {
                    "event_id": event_id,
                    "time_ut": parts[0],
                    "north_lat": coordinates[0],
                    "north_lon": coordinates[1],
                    "south_lat": coordinates[2],
                    "south_lon": coordinates[3],
                    "center_lat": coordinates[4],
                    "center_lon": coordinates[5],
                    "diameter_ratio": pd.to_numeric(extra[0], errors="coerce"),
                    "sun_alt": pd.to_numeric(extra[1], errors="coerce"),
                    "sun_azm": pd.to_numeric(extra[2], errors="coerce"),
                    "path_width_km": pd.to_numeric(extra[3], errors="coerce"),
                    "central_duration": extra[4]
                }

                rows.append(row)

            except (ValueError, IndexError):
                continue

    return pd.DataFrame(rows)

The path index links to a separate page for each event. We retain its identifier on every timed row so we can reconstruct individual tracks after concatenation. Only rows beginning with a clock time enter the comparison.

In [ ]:
# 4. Find the eclipse path links
response = requests.get(url_c, timeout=60)
response.raise_for_status()

soup = BeautifulSoup(response.content, "html.parser")

path_urls = []

for link in soup.find_all("a", href=True):  # Since the webpage has multiple tables that we need to access through different links.

    if link.get_text(strip=True).lower() == "path":
        path_url = urljoin(url_c, link["href"])
        path_urls.append(path_url)

path_urls = list(dict.fromkeys(path_urls))

print("Paths found:", len(path_urls))


# 5. Extract all eclipse paths
path_data = []

for i, url in enumerate(path_urls, 1):

    try:
        df_path = extract_path(url)

        if not df_path.empty:
            path_data.append(df_path)
        else:
            print("No data:", url)

    except requests.RequestException as error:
        print("Error:", url, error)

    print(f"Processed: {i}/{len(path_urls)}")
    time.sleep(0.2)


# 6. Combine and save the dataset
if not path_data:
    raise ValueError("No eclipse paths were extracted.")

df_c = pd.concat(path_data, ignore_index=True)

df_c.to_csv("dataset_C.csv", index=False)

print("Dataset C:", df_c.shape)
print("Eclipses extracted:", df_c["event_id"].nunique())

display(df_c.head(20))

## D · Which positions belong to a path point?

A time such as 00:02 is ambiguous on its own. If it follows 23:58 in the same trajectory, it belongs to the following day. We count backward jumps in the clock and align the middle observation with the event date before requesting vectors.

In [ ]:
import pandas as pd
import requests
import time
from io import StringIO

# =========================================================
# DATASET D — JPL HORIZONS API
# =========================================================

# 1. Load Dataset C
df_c = pd.read_csv("dataset_C.csv")

url_d = "https://ssd.jpl.nasa.gov/api/horizons.api"


# 2. Add complete dates to Dataset C
events = []

for event_id, group in df_c.groupby("event_id", sort=False):

    group = group.copy()

    # Get the eclipse date from event_id
    event_date = pd.to_datetime(
        event_id[2:11], format="%Y%b%d"
    )

    # Convert HH:MM into time values
    clock = pd.to_timedelta(group["time_ut"] + ":00")

    # Detect when the trajectory crosses midnight
    new_day = (clock.diff() < pd.Timedelta(0)).cumsum()

    full_time = (
        event_date
        + clock
        + pd.to_timedelta(new_day, unit="D")
    )

    # Align the middle of the trajectory with the event date
    middle_date = full_time.iloc[len(group) // 2].normalize()

    full_time = full_time + (event_date - middle_date)

    group["datetime_ut"] = full_time

    events.append(group)

df_c = pd.concat(events, ignore_index=True)

Horizons returns a geocentric state $(x,y,z,v_x,v_y,v_z)$ for each body. Positions are in km and velocities in km/s along ICRF axes. These are geometric vectors. The separate light-time calculation belongs to the trajectory model.

In [ ]:
# 3. Extract vectors from JPL Horizons
def get_horizons(body_id, start, stop):

    params = {
        "format": "text",
        "COMMAND": f"'{body_id}'",
        "OBJ_DATA": "'NO'",
        "MAKE_EPHEM": "'YES'",
        "EPHEM_TYPE": "'VECTORS'",
        "CENTER": "'500@399'",
        "REF_PLANE": "'FRAME'",
        "REF_SYSTEM": "'ICRF'",
        "OUT_UNITS": "'KM-S'",
        "VEC_TABLE": "'2'",
        "VEC_LABELS": "'NO'",
        "VEC_CORR": "'NONE'",
        "CSV_FORMAT": "'YES'",
        "TIME_TYPE": "'UT'",
        "TIME_DIGITS": "'MINUTES'",
        "STEP_SIZE": "'1 m'",
        "START_TIME": f"'{start}'",
        "STOP_TIME": f"'{stop}'"
    }

    response = requests.get(
        url_d, params=params, timeout=90
    )

    response.raise_for_status()

    text = response.text

    # 4. Extract the data section
    if "$$SOE" not in text or "$$EOE" not in text:
        raise ValueError("No ephemeris data found: " + text[:300])

    data = text.split("$$SOE")[1]
    data = data.split("$$EOE")[0]

    # 5. Create the DataFrame
    columns = [
        "jd_ut", "date_ut",
        "x_km", "y_km", "z_km",
        "vx_kms", "vy_kms", "vz_kms"
    ]

    df = pd.read_csv(
        StringIO(data),
        header=None,
        names=columns,
        usecols=range(8),
        skipinitialspace=True
    )

    # 6. Convert the dates
    df["date_ut"] = df["date_ut"].str.replace(
        "A.D.", "", regex=False
    )

    df["date_ut"] = df["date_ut"].str.strip()

    df["datetime_ut"] = pd.to_datetime(
        df["date_ut"],
        format="mixed"
    )

    # 7. Convert numerical columns
    for col in columns:
        if col != "date_ut":
            df[col] = pd.to_numeric(df[col], errors="raise")

    # 8. Verify timestamps have minute precision
    if not (df["datetime_ut"] == df["datetime_ut"].dt.floor("min")).all():
        raise ValueError("Unexpected timestamp precision.")

    # 9. Remove unnecessary columns
    df = df.drop(columns=["jd_ut", "date_ut"])

    return df

We query each body at one-minute intervals, then select the times actually present in C. The one-to-one join checks that a solar state, a lunar state and a path observation all describe the same timestamp.

In [ ]:
# 10. Extract Sun and Moon vectors for every eclipse
path_data = []

for event_id, group in df_c.groupby("event_id", sort=False):

    # Observation interval
    start = group["datetime_ut"].min()
    stop = group["datetime_ut"].max() + pd.Timedelta(minutes=1)

    start = start.strftime("%Y-%m-%d %H:%M")
    stop = stop.strftime("%Y-%m-%d %H:%M")

    # Extract Sun and Moon data
    df_sun = get_horizons("10", start, stop)
    df_moon = get_horizons("301", start, stop)

    # Combine both bodies at the same time
    df_vectors = pd.merge(
        df_sun,
        df_moon,
        on="datetime_ut",
        suffixes=("_sun", "_moon"),
        validate="one_to_one"
    )

    # Keep the exact observations from Dataset C
    df_event = pd.merge(
        group[["event_id", "time_ut", "datetime_ut"]],
        df_vectors,
        on="datetime_ut",
        how="left",
        validate="one_to_one"
    )

    # Verify the extraction
    if df_event.isnull().any().any():
        raise ValueError(f"Missing observations: {event_id}")

    path_data.append(df_event)

    print("Completed:", event_id)

    time.sleep(0.2)


# 11. Combine all eclipses
if not path_data:
    raise ValueError("No eclipse data was extracted.")

df_d = pd.concat(path_data, ignore_index=True)


# 12. Final validation
if len(df_d) != len(df_c):
    raise ValueError("Dataset D does not match Dataset C.")

if df_d["event_id"].nunique() != df_c["event_id"].nunique():
    raise ValueError("Some eclipses are missing.")

if df_d.duplicated(["event_id", "datetime_ut"]).any():
    raise ValueError("Duplicated observations found.")


# 13. Save Dataset D
df_d.to_csv("dataset_D.csv", index=False)

print("Dataset D:", df_d.shape)
print("Unique eclipses:", df_d["event_id"].nunique())

display(df_d.head())

Dataset A supplies the labels, C supplies the reference central line, and D supplies the states at its timed observations. Consequently a date missing from C can still be a positive example in A.

Eclipse predictions and Besselian elements are credited to Fred Espenak and Jean Meeus, NASA/GSFC. Ephemerides are from the NASA/JPL Horizons system. [NASA eclipse catalog](https://eclipse.gsfc.nasa.gov/SEpubs/5MKSE.html) · [NASA path tables](https://eclipse.gsfc.nasa.gov/SEpath/SEpath.html) · [JPL Horizons](https://ssd.jpl.nasa.gov/horizons/)